# 🧠 Delta Prediction Strategy — Break Persistence Bias
**Idea:** Predict `ΔG = Future_CGM − Current_CGM` instead of absolute glucose.  
**Reconstruct:** `Predicted_Absolute = CGM + Predicted_Delta`  
**Goal:** Reduce CGM feature dominance → lower Zone D → Clarke A+B > 95%

---

In [ ]:
!pip install polars xgboost plotly pyarrow scikit-learn -q

import polars as pl, numpy as np, xgboost as xgb
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime, timedelta
import os, gc, time, json

from google.colab import drive
drive.mount('/content/drive')
print(f"XGBoost: {xgb.__version__}")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
BASE_PATH  = "/content/drive/MyDrive/DiabetesKDS/data"
INPUT_FILE = f"{BASE_PATH}/processed/train_elite_physio_engineered.parquet"
MODEL_FILE = f"{BASE_PATH}/models/xgboost_delta_v3.json"
os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)

FEATURES = [
    "CGM", "IOB", "COB", "Vel_5min", "Vel_15min", "Acc_15min",
    "Roll_Mean_60", "Roll_Std_60", "Hour_Sin", "Hour_Cos",
    "Insulin_Sensitivity_Risk",
]
print(f"📁 {INPUT_FILE} — exists: {os.path.exists(INPUT_FILE)}")

---\n## STEP 1 — Data Prep: Create Delta Target

In [ ]:
df = pl.read_parquet(INPUT_FILE)
FEATURES = [c for c in FEATURES if c in df.columns]
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

# Create Delta target
df = df.with_columns(
    (pl.col("Target_CGM") - pl.col("CGM")).alias("Target_Delta")
)

# Drop rows with nulls in features or targets
df = df.sort("datetime").drop_nulls(subset=FEATURES + ["Target_Delta", "Target_CGM"])
print(f"After cleanup: {len(df):,} rows")

# Delta statistics
delta = df["Target_Delta"]
print(f"\nTarget_Delta stats:")
print(f"  Min:    {delta.min():.2f}")
print(f"  Max:    {delta.max():.2f}")
print(f"  Mean:   {delta.mean():.4f}")
print(f"  Std:    {delta.std():.2f}")
print(f"  Median: {delta.median():.2f}")

In [ ]:
# Delta distribution
fig = px.histogram(
    df.select(pl.col("Target_Delta").clip(-80, 80)).to_pandas(),
    x="Target_Delta", nbins=200,
    title="📊 Target_Delta Distribution (CGM change in 30 min)",
    color_discrete_sequence=["#4ECDC4"],
)
fig.add_vline(x=0, line_dash="dash", line_color="white", annotation_text="No change")
fig.update_layout(template="plotly_dark", height=350)
fig.show()

In [ ]:
# ── Time-Series Split ────────────────────────────────────────────────
split = int(len(df) * 0.80)
df_train, df_test = df.head(split), df.tail(len(df) - split)

X_train = df_train.select(FEATURES).to_numpy().astype(np.float32)
y_train_delta = df_train["Target_Delta"].to_numpy().astype(np.float32)
y_train_abs   = df_train["Target_CGM"].to_numpy().astype(np.float32)
cgm_train     = df_train["CGM"].to_numpy().astype(np.float32)

X_test  = df_test.select(FEATURES).to_numpy().astype(np.float32)
y_test_delta = df_test["Target_Delta"].to_numpy().astype(np.float32)
y_test_abs   = df_test["Target_CGM"].to_numpy().astype(np.float32)
cgm_test     = df_test["CGM"].to_numpy().astype(np.float32)

test_ids = df_test["id"].to_list()
test_dt  = df_test["datetime"].to_list()

del df, df_train; gc.collect()
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

---\n## STEP 2 — Clinical Weights (Based on Absolute Glucose)

In [ ]:
def clinical_weights(y_abs):
    w = np.ones_like(y_abs, dtype=np.float32)
    w[y_abs < 70]                     = 10.0
    w[y_abs > 250]                    = 5.0
    w[(y_abs >= 180) & (y_abs <= 250)] = 2.0
    return w

w_train = clinical_weights(y_train_abs)
w_test  = clinical_weights(y_test_abs)

for lbl, cond in [("Hypo <70 (10x)", w_train==10), ("Hyper >250 (5x)", w_train==5),
                   ("Mild 180-250 (2x)", w_train==2), ("Normal (1x)", w_train==1)]:
    print(f"  {lbl:22s}: {cond.sum():>10,} ({cond.sum()/len(w_train)*100:.2f}%)")

---\n## STEP 3 — Train Delta Model (T4 Optimized) 🚀

In [ ]:
model = xgb.XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist", device="cuda",
    n_estimators=3000, early_stopping_rounds=50,
    eval_metric="rmse", random_state=42, verbosity=1,
    # Optuna-derived params
    learning_rate=0.015, max_depth=12,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.8, reg_lambda=8.1, min_child_weight=16,
)

print("🚀 Training Delta model...")
t0 = time.time()

model.fit(
    X_train, y_train_delta,
    sample_weight=w_train,
    eval_set=[(X_train, y_train_delta), (X_test, y_test_delta)],
    sample_weight_eval_set=[w_train, w_test],
    verbose=200,
)

t_train = time.time() - t0
print(f"\n✅ Done in {t_train:.1f}s | Best iter: {model.best_iteration}")

In [ ]:
# Learning curves
res = model.evals_result()
fig = go.Figure()
fig.add_trace(go.Scatter(y=res["validation_0"]["rmse"], name="Train", line=dict(color="#4ECDC4")))
fig.add_trace(go.Scatter(y=res["validation_1"]["rmse"], name="Test", line=dict(color="#FF6B6B")))
fig.add_vline(x=model.best_iteration, line_dash="dash", line_color="white")
fig.update_layout(title=f"📉 Delta Model Learning Curves (best={model.best_iteration})",
    template="plotly_dark", height=380, xaxis_title="Round", yaxis_title="RMSE")
fig.show()

---\n## STEP 4 — Reconstruct & Evaluate 🔬

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RECONSTRUCT: Predicted_Absolute = CGM + Predicted_Delta
# ══════════════════════════════════════════════════════════════════════
pred_delta = model.predict(X_test)
pred_abs   = np.clip(cgm_test + pred_delta, 20, 500)

# Also compute delta-level metrics
rmse_delta = np.sqrt(mean_squared_error(y_test_delta, pred_delta))
mae_delta  = mean_absolute_error(y_test_delta, pred_delta)

# Absolute-level metrics (what Clarke Grid uses)
rmse_abs = np.sqrt(mean_squared_error(y_test_abs, pred_abs))
mae_abs  = mean_absolute_error(y_test_abs, pred_abs)
mask = y_test_abs > 0
mape = np.mean(np.abs((y_test_abs[mask] - pred_abs[mask]) / y_test_abs[mask])) * 100

errs = np.abs(y_test_abs - pred_abs)
w10 = np.mean(errs <= 10) * 100
w15 = np.mean(errs <= 15) * 100
w20 = np.mean(errs <= 20) * 100

print("╔═══════════════════════════════════════════════════════╗")
print("║         📊  DELTA MODEL — TEST METRICS               ║")
print("╠═══════════════════════════════════════════════════════╣")
print(f"║  DELTA RMSE   : {rmse_delta:>8.4f} mg/dL                  ║")
print(f"║  DELTA MAE    : {mae_delta:>8.4f} mg/dL                  ║")
print(f"║  ─────────────────────────────────────────────────   ║")
print(f"║  ABSOLUTE RMSE: {rmse_abs:>8.4f} mg/dL (reconstructed)    ║")
print(f"║  ABSOLUTE MAE : {mae_abs:>8.4f} mg/dL                    ║")
print(f"║  MAPE         : {mape:>8.4f}%                           ║")
print(f"║  Within ±10   : {w10:>8.2f}%                             ║")
print(f"║  Within ±15   : {w15:>8.2f}%                             ║")
print(f"║  Within ±20   : {w20:>8.2f}%                             ║")
print("╚═══════════════════════════════════════════════════════╝")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CLARKE ERROR GRID
# ══════════════════════════════════════════════════════════════════════
def clarke_zones(ref, pred):
    ref, pred = np.asarray(ref, float), np.asarray(pred, float)
    z = np.full(len(ref), 'D', dtype='U1')
    for i in range(len(ref)):
        r, p = ref[i], pred[i]
        if (r <= 70 and p <= 70) or abs(p - r) <= 0.2 * r:
            z[i] = 'A'
        elif (r <= 70 and p >= 180) or (r >= 180 and p <= 70):
            z[i] = 'E'
        elif 70 <= r <= 180 and (p < 70 or p > 180):
            z[i] = 'C'
        elif (r < 70 and 70 < p <= 180) or (r > 180 and 70 <= p < 180):
            z[i] = 'D'
        else:
            z[i] = 'B'
    return z

zones = clarke_zones(y_test_abs, pred_abs)
uq, cts = np.unique(zones, return_counts=True)
zs = dict(zip(uq, cts))
total = len(zones)
zone_ab = (zs.get('A',0) + zs.get('B',0)) / total * 100

print("╔═══════════════════════════════════════════════════╗")
print("║       🩸  CLARKE ERROR GRID (Delta v3)            ║")
print("╠═══════════════════════════════════════════════════╣")
for zone in ['A','B','C','D','E']:
    n = zs.get(zone, 0); pct = n/total*100
    bar = '█' * int(pct/2)
    print(f"║  Zone {zone}: {n:>10,} ({pct:>6.2f}%) {bar:20s}  ║")
print(f"║                                                   ║")
s = "✅ PASS" if zone_ab > 95 else "⚠️  BELOW"
print(f"║  A+B = {zone_ab:.2f}%  (target >95%)  {s:12s}    ║")
print(f"║  v1=92.91%  →  Δ = {zone_ab-92.91:+.2f}%                    ║")
print("╚═══════════════════════════════════════════════════╝")

In [ ]:
# Clarke Error Grid Plot
sn = min(100000, total)
idx = np.random.RandomState(42).choice(total, sn, replace=False)
z_s = clarke_zones(y_test_abs[idx], pred_abs[idx])
clr = {'A':'#2ECC71','B':'#F39C12','C':'#E74C3C','D':'#9B59B6','E':'#C0392B'}

fig = go.Figure()
fig.add_trace(go.Scattergl(x=y_test_abs[idx], y=pred_abs[idx], mode='markers',
    marker=dict(size=2, color=[clr.get(z,'gray') for z in z_s], opacity=0.4), hoverinfo='skip'))
fig.add_trace(go.Scatter(x=[0,400], y=[0,400], mode='lines',
    line=dict(color='white', width=1, dash='dash'), name='Perfect'))
x_l = np.linspace(1,400,300)
fig.add_trace(go.Scatter(x=x_l, y=x_l*1.2, mode='lines',
    line=dict(color='rgba(46,204,113,0.4)'), showlegend=False))
fig.add_trace(go.Scatter(x=x_l, y=x_l*0.8, mode='lines',
    line=dict(color='rgba(46,204,113,0.4)'), showlegend=False))
for v in [70,180]:
    fig.add_hline(y=v, line_dash='dot', line_color='rgba(255,255,255,0.2)')
    fig.add_vline(x=v, line_dash='dot', line_color='rgba(255,255,255,0.2)')
fig.add_annotation(x=340, y=50, text=f"A+B: {zone_ab:.1f}%",
    font=dict(size=16, color='white'), showarrow=False)
fig.update_layout(title="🩸 Clarke Grid — Delta Strategy v3",
    xaxis_title="Reference (mg/dL)", yaxis_title="Predicted (mg/dL)",
    template="plotly_dark", height=600, width=600,
    xaxis=dict(range=[0,400]), yaxis=dict(range=[0,400]))
fig.show()

---\n## STEP 5 — Feature Importance (Persistence Bias Check)

In [ ]:
imp = model.get_booster().get_score(importance_type="gain")
fi = {FEATURES[int(k.replace('f',''))]: v for k, v in imp.items() if int(k.replace('f','')) < len(FEATURES)}
fi = dict(sorted(fi.items(), key=lambda x: x[1], reverse=True))
total_gain = sum(fi.values())

print("╔══════════════════════════════════════════════════════════╗")
print("║      🏆  FEATURE IMPORTANCE (Delta Model)              ║")
print("╠══════════════════════════════════════════════════════════╣")
for name, val in fi.items():
    pct = val / total_gain * 100
    bar = '█' * int(pct)
    flag = " ⚠️" if name == 'CGM' and pct > 50 else ""
    print(f"║  {name:28s} {pct:>6.2f}%  {bar:20s}{flag}  ║")
print("╚══════════════════════════════════════════════════════════╝")

cgm_pct = fi.get('CGM', 0) / total_gain * 100
print(f"\n🔍 CGM share: {cgm_pct:.1f}%")
print(f"   v1 (absolute target): ~80% CGM dominance")
print(f"   v3 (delta target):    {cgm_pct:.1f}% {'✅ Bias broken!' if cgm_pct < 50 else '⚠️ Still dominant'}")

fig = px.bar(x=list(fi.values()), y=list(fi.keys()), orientation='h',
    title='🏆 Feature Importance — Delta v3', color=list(fi.values()),
    color_continuous_scale='Viridis')
fig.update_layout(template='plotly_dark', height=400, yaxis=dict(autorange='reversed'))
fig.show()

In [ ]:
# ── 24h Prediction Trace ─────────────────────────────────────────────
tid = np.array(test_ids)
uid, uc = np.unique(tid, return_counts=True)
best_p = uid[np.argmax(uc)]
pm = tid == best_p
pdt = np.array(test_dt)[pm]

mid = len(pdt)//2
wm = (pdt >= pdt[mid]) & (pdt <= pdt[mid] + timedelta(hours=24))

fig = go.Figure()
fig.add_trace(go.Scatter(x=pdt[wm], y=y_test_abs[pm][wm], name='Actual',
    line=dict(color='#00D4AA', width=2)))
fig.add_trace(go.Scatter(x=pdt[wm], y=pred_abs[pm][wm], name='Predicted (Delta→Abs)',
    line=dict(color='#FF6B6B', width=2, dash='dot')))
fig.add_hline(y=70, line_dash='dash', line_color='red')
fig.add_hline(y=180, line_dash='dash', line_color='orange')
fig.update_layout(title=f"📉 24h Trace — Patient '{best_p}' (Delta Reconstruction)",
    template='plotly_dark', height=420, xaxis_title='Time', yaxis_title='mg/dL')
fig.show()

---\n## STEP 6 — Save & Final Verdict

In [ ]:
model.save_model(MODEL_FILE)
msz = os.path.getsize(MODEL_FILE) / (1024**2)

meta = {
    "model": "xgboost_delta_v3", "strategy": "delta_prediction",
    "created": datetime.now().isoformat(),
    "features": FEATURES, "target": "Target_Delta (=Target_CGM - CGM)",
    "reconstruction": "Predicted_Absolute = CGM + Predicted_Delta",
    "horizon_min": 30, "best_iteration": int(model.best_iteration),
    "metrics": {
        "delta_rmse": round(float(rmse_delta),4),
        "absolute_rmse": round(float(rmse_abs),4),
        "absolute_mae": round(float(mae_abs),4),
        "mape": round(float(mape),4),
        "within_10": round(float(w10),2),
        "clarke_a_pct": round(zs.get('A',0)/total*100, 2),
        "clarke_ab_pct": round(float(zone_ab), 2),
    },
    "cgm_importance_pct": round(float(cgm_pct), 2),
    "hyperparameters": {
        "learning_rate": 0.015, "max_depth": 12,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "reg_alpha": 1.8, "reg_lambda": 8.1, "min_child_weight": 16,
    },
}
with open(MODEL_FILE.replace('.json','_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f"💾 Model: {MODEL_FILE} ({msz:.1f} MB)")

In [ ]:
print("")
print("╔═══════════════════════════════════════════════════════════════════╗")
print("║           🏁  MODEL EVOLUTION REPORT                            ║")
print("╠═══════════════════════════════════════════════════════════════════╣")
print("║                     v1 (abs)    v2 (optuna)   v3 (delta)        ║")
print("║  ───────────────────────────────────────────────────────────     ║")
print(f"║  Clarke A+B:       92.91%        —          {zone_ab:>6.2f}%          ║")
print(f"║  Zone D:            4.80%        —          {zs.get('D',0)/total*100:>6.2f}%          ║")
print(f"║  CGM dominance:      ~80%        —          {cgm_pct:>6.1f}%          ║")
print(f"║  RMSE (abs):          —          —          {rmse_abs:>8.4f}          ║")
print(f"║  MAE (abs):           —          —          {mae_abs:>8.4f}          ║")
print("║                                                                 ║")
print(f"║  Strategy:   Predict ΔG, reconstruct: CGM + ΔG_pred           ║")
print(f"║  Weights:    Hypo=10x, Hyper=5x, Mild=2x                      ║")
print(f"║  Best Round: {model.best_iteration:>6} / 3000                               ║")
print("║                                                                 ║")
v = "✅ PERSISTENCE BIAS BROKEN" if cgm_pct < 50 else "⚠️  CGM still dominant"
print(f"║  BIAS:    {v:55s} ║")
v2 = "✅ TARGET ACHIEVED" if zone_ab > 95 else "⚠️  Below 95% target"
print(f"║  CLARKE:  {v2:55s} ║")
print("╚═══════════════════════════════════════════════════════════════════╝")

gc.collect()
print("\n🧹 Done.")